In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Liza's cross match code

def get_tile(ra: float, dec: float, tiles_df: pd.DataFrame) -> tuple[str, float, float]:
    """
    Returns the closest CFIS tile from tiles_df to the galaxy through
        comparison of ra and dec, returns 'No tiles found' if no good 
        tiles exist.
    
    Parameters:
        ra (float): Right ascension coordinate of galaxy [degrees]
        dec (float): Declination coordinate of galaxy [degrees]
        tiles_df (pandas.DataFrame): Datafram of CFIS tile containing 
            at least the columns "tile", "ra" and "dec"
    
    Returns:
        A tuple of (tile_name, tile_ra, and tile_dec) of the closest tile.
        "No tile found" if no suitable tile is found.
    """

    ra_dist = np.abs(tiles_df.ra - ra)
    ra_dist[ra_dist > 180] = 360-ra_dist[ra_dist>180]
    dec_dist = np.abs(tiles_df.dec - dec)

    dec_thresh = 0.25
    ra_thresh = 0.25/np.cos(dec*np.pi/180)

    good_tiles = np.argwhere((ra_dist < ra_thresh) & (dec_dist < dec_thresh))
    if len(good_tiles) == 0:
         return ("No tile found", np.nan, np.nan)
    else:
        dist = ra_dist[good_tiles[:,0]]**2 + dec_dist[good_tiles[:,0]]**2
        tile_row = tiles_df.iloc[dist.idxmin()]
        return (tile_row["tile"], tile_row["ra"], tile_row["dec"])
    

def cross_match(galaxies_file: str, tiles_file: str, output_file: str) -> pd.DataFrame:
    """
    Cross-match galaxies from galaxies_file to tiles in tiles_file and
        outputs the matching galaxies and tile results to output_file

    Parameters: 
        galaxies_file (str): CSV file containing galaxies to be matched
        tiles_file (str): CSV file containing tiles to be matched
        output_file (str): CSV file where output is written to
    
    Returns:
        pandas.Dataframe of the galaxies with three additional columns 
            containing information of the matched tile.
    """
    galaxies_df = pd.read_csv(galaxies_file)
    tiles_df = pd.read_csv(tiles_file)

    galaxies_df[["tile_name", "tile_ra", "tile_dec"]] = galaxies_df.apply(
        lambda row: pd.Series(get_tile(row["ra"], row["dec"], tiles_df)),
        axis=1
    )

    galaxies_df.to_csv(output_file, index=False)

    return galaxies_df

cross_match('sdssquery.csv', 'cfis_r_tiles.csv', 'sdssunions.csv')